In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig
from src.iaaft import surrogates

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Isolate one case

In [5]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
x = torch.tensor([0.2])
y = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    x_next = x[t] * (3.78 - (3.78 * x[t]))
    y_next = y[t] * (3.77 - (3.77 * y[t]) - (0.8 * x[t]))

    x = torch.concat((x, x_next.unsqueeze(0)))
    y = torch.concat((y, y_next.unsqueeze(0)))
            
# Normalising step
x_norm = x.sub(x.mean(dim = -1).unsqueeze(-1)).div(x.std(dim = -1).unsqueeze(-1))
y_norm = y.sub(y.mean(dim = -1).unsqueeze(-1)).div(y.std(dim = -1).unsqueeze(-1))

In [43]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
x = torch.tensor([0.2])
y = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    # from ECCM
    x_next = x[t] * (3.8 - (3.8 * x[t]))
    y_next = y[t] * (3.1 - (3.1 * y[t]) - (0.8 * x[t]))

    x = torch.concat((x, x_next.unsqueeze(0)))
    y = torch.concat((y, y_next.unsqueeze(0)))
            
# Normalising step
x_norm = x.sub(x.mean(dim = -1).unsqueeze(-1)).div(x.std(dim = -1).unsqueeze(-1))
y_norm = y.sub(y.mean(dim = -1).unsqueeze(-1)).div(y.std(dim = -1).unsqueeze(-1))

In [44]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, x_norm.shape[0]), y = x_norm,
                    mode = 'lines+markers',
                    name = 'X'))

fig.add_trace(go.Scatter(x = torch.arange(0, x_norm.shape[0]), y = y_norm,
                    mode = 'lines+markers',
                    name = 'Y'))

fig.update_layout(title = 'Confounding time series"',
                   xaxis_title = 'epoch',
                   yaxis_title = 'loss')

fig.show()

In [45]:
y_norm_iaaft = torch.tensor(surrogates(x = y_norm, ns = 1), dtype = torch.float32).squeeze()
x_norm_iaaft = torch.tensor(surrogates(x = x_norm, ns = 1), dtype = torch.float32).squeeze()

y_norm_iaaft_100 = torch.tensor(surrogates(x = y_norm, ns = 100, tol_pc = 5.), dtype = torch.float32).squeeze()
x_norm_iaaft_100 = torch.tensor(surrogates(x = x_norm, ns = 100, tol_pc = 5.), dtype = torch.float32).squeeze()

Estim100%|██████████████████████████████| 100/100 [00:00<00:00, 3133.61it/s]


In [48]:
### FIXED FILTER
k = 3
max_offset_sig = torch.tensor([k - 1]).to(device) # allow instantanous 
# max_offset_sig = torch.tensor([-1]).to(device)
sig_filter = torch.ones(size = (k, )).to(device)

N_length = x_norm.shape[0]
N = N_length - k + 1
l_train_masks_L100 = generate_mask_tensor(N, 100)

noise_scalar = torch.tensor([0.05], device = device)

In [49]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Y -> X
                           x = y_norm.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

rho_l_L100 = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
    rho_L100, nlml_L100 = GP_ccm_sig(
            y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
            y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
            x_train = x_gt[l_train_masks_L100[l]].to(device),
            x_test = x_gt[ ~ l_train_masks_L100[l]].to(device),
            noise = noise_scalar,
            rbf_sigma = 0.5,
            device = device)
            
    rho_l_L100 = torch.concat((rho_l_L100, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

print(rho_l_L100.mean().item())
print(rho_l_L100.std().item())

IndexError: The shape of the mask [398] at index 0 does not match the shape of the indexed tensor [0] at index 0

In [37]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm_iaaft.to(device), # Testing X xmap Y, thus Y -> X
                           x = y_norm.to(device),
                           # x = y_norm_iaaft.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

rho_l_L100 = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
    rho_L100, nlml_L100 = GP_ccm_sig(
            y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
            y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
            x_train = x_gt[l_train_masks_L100[l]].to(device),
            x_test = x_gt[ ~ l_train_masks_L100[l]].to(device),
            noise = noise_scalar,
            rbf_sigma = 0.5,
            device = device)
            
    rho_l_L100 = torch.concat((rho_l_L100, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

print(rho_l_L100.mean().item())
print(rho_l_L100.std().item())

0.006930300500243902
0.05554981529712677


In [38]:
# 3 minutes
rho_collector = torch.empty(size = (1, N)).to(device)

for i in range(y_norm_iaaft_100.shape[0]):
    y_norm_iaaft = y_norm_iaaft_100[i]

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing X xmap Y, thus Y -> X
                           x = y_norm_iaaft.to(device),
                           # x = y_norm_iaaft.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    rho_l_L100 = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho_L100, nlml_L100 = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks_L100[l]].to(device),
                x_test = x_gt[ ~ l_train_masks_L100[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.5,
                device = device)
            
        rho_l_L100 = torch.concat((rho_l_L100, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

    rho_collector = torch.concat((rho_collector, rho_l_L100), dim = 0)

In [41]:
q = torch.tensor([0.01, 0.99]).to(device)
q = torch.tensor([0.05, 0.95]).to(device)
torch.quantile(rho_collector, q)

tensor([-0.0940,  0.1122], device='cuda:0')

# X -> Y (true)

In [46]:
shifts = torch.arange(- 8, 8 + 1, 1)
print(shifts)

# fix filter
k = 2 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())
    print(i)

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing X -> Y
                           x = x_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for X -> Y',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])
-8
0
-7
1
-6
2
-5
3
-4
4
-3
5
-2
6
-1
7
0
8
1
9
2
10
3
11
4
12
5
13
6
14
7
15
8
16


# Y -> X (false)

In [47]:
shifts = torch.arange(- 8, 8 + 1, 1)
print(shifts)

# fix filter
k = 3 # 2 best
sig_filter = torch.ones(size = (k, )).to(device)

noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())
    print(i)

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Y -> X
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Y -> X (false)',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-8, -7, -6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8])
-8
0
-7
1
-6
2
-5
3
-4
4
-3
5
-2
6
-1
7
0
8
1
9
2
10
3
11
4
12
5
13
6
14
7
15
8
16


In [45]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing X -> Y
                           x = x_norm.to(device),
                           max_pos_offset = max_offset_sig, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

rho_l_L100 = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
    rho_L100, nlml_L100 = GP_ccm_sig(
            y_embeddings_train = y_embeddings[l_train_masks_L100[l]].unsqueeze(-1).to(device),
            y_embeddings_test = y_embeddings[ ~ l_train_masks_L100[l]].unsqueeze(-1).to(device),
            x_train = x_gt[l_train_masks_L100[l]].to(device),
            x_test = x_gt[ ~ l_train_masks_L100[l]].to(device),
            noise = noise_scalar,
            rbf_sigma = 0.05,
            device = device)
            
    rho_l_L100 = torch.concat((rho_l_L100, rho_L100.unsqueeze(0).unsqueeze(0)), dim = 1)

rho_l_L100.mean()

tensor(0.8423, device='cuda:0')